<a href="https://colab.research.google.com/github/Pranayshukla0610/ML-projects-portfolio/blob/main/Gen_AI_Shoes_Generator_through_Diffusion_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install diffusers transformers accelerate safetensors gradio

In [ ]:
import torch
import matplotlib.pyplot as plt
from diffusers import (
    StableDiffusionPipeline,
    DPMSolverMultistepScheduler
)
import gradio as gr

In [ ]:
device = 'cpu'
print(device)

In [ ]:
model_id = "runwayml/stable-diffusion-v1-5"


pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float32
)


pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config
)


pipe.to(device)

In [ ]:
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

In [ ]:
def add_noise(latent, noise, timestep, scheduler):
  noisy_latent = scheduler.add_noise(
      latent,
      noise,
      timestep
  )

  return noisy_latent

In [ ]:
prompt = """
A futuristic running shoe,
metallic design,
carbon fiber sole,
premium sports footwear,
studio lighting
"""

In [ ]:
text_embeddings = pipe.encode_prompt(
    prompt=prompt,
    device=device,
    num_images_per_prompt=1,
    do_classifier_free_guidance=False
)

text_embeddings

In [ ]:
prompt = """
A luxury futuristic sneaker,
black and red color,
carbon fiber texture,
high performance running shoe,
realistic product photography
"""

In [ ]:
image = pipe(
    prompt,
    height=256,
    width=256,
    num_inference_steps=25,
    guidance_scale=7.5
).images[0]

In [ ]:
def generate_shoe(prompt):

    image = pipe(
        prompt,
        height=256,
        width=256,
        num_inference_steps=20,
        guidance_scale=7
    ).images[0]


    return image

In [ ]:
prompts=[
"Futuristic basketball shoe with LED lights",
"Luxury leather formal shoe with golden texture",
"Eco friendly sneaker made from recycled material",
"Cyberpunk running shoe with neon colors"
]


for p in prompts:

    img=generate_shoe(p)

    plt.figure(figsize=(3,3))
    plt.imshow(img)
    plt.title(p)
    plt.axis("off")
    plt.show()

In [ ]:
interface = gr.Interface(

    fn=generate_shoe,

    inputs=
    gr.Textbox(
        label="Describe your shoe design"
    ),

    outputs=
    gr.Image(
        label="Generated Shoe"
    ),

    title="AI Shoe Design Generator",

    description=
    """
    Generate customized footwear designs
    using Latent Diffusion Model.
    """

)


interface.launch()